In [94]:
from google.colab import files
import pandas as pd
import numpy as np
from itertools import combinations
from collections import defaultdict
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [95]:
uploaded = files.upload()

Saving data_clean.csv to data_clean (2).csv


In [96]:
df = pd.read_csv("data_clean.csv", encoding="latin1", low_memory=False)

In [97]:
df.head()

,InvoiceNo,StockCode,Description,Quantity,UnitPrice,CustomerID,Country,Revenue,InvoiceDate_ISO
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2.55,17850,United Kingdom,15.30,2010-12-01 08:26:00
1,536365,71053,WHITE METAL LANTERN,6,3.39,17850,United Kingdom,20.34,2010-12-01 08:26:00
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2.75,17850,United Kingdom,22.00,2010-12-01 08:26:00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,3.39,17850,United Kingdom,20.34,2010-12-01 08:26:00
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,3.39,17850,United Kingdom,20.34,2010-12-01 08:26:00


In [98]:
df["InvoiceDate_ISO"] = pd.to_datetime(df["InvoiceDate_ISO"])
df["CustomerID"] = df["CustomerID"].astype(int)


In [99]:
print("Loaded:", df.shape)

Loaded: (397884, 9)


In [100]:
order_revenue = df.groupby("InvoiceNo")["Revenue"].sum()
kpi = pd.DataFrame([{
    "total_revenue":            round(df["Revenue"].sum(), 2),
    "total_customers":          df["CustomerID"].nunique(),
    "total_orders":             df["InvoiceNo"].nunique(),
    "avg_order_value":          round(order_revenue.mean(), 2),
    "avg_revenue_per_customer": round(df.groupby("CustomerID")["Revenue"].sum().mean(), 2)
}])
print("\nKPI done")


KPI done


In [101]:
df["month"] = df["InvoiceDate_ISO"].dt.to_period("M").astype(str)
monthly = df.groupby("month").agg(
    total_revenue   =("Revenue",    "sum"),
    total_orders    =("InvoiceNo",  "nunique"),
    total_customers =("CustomerID", "nunique")
).reset_index()
monthly["total_revenue"] = monthly["total_revenue"].round(2)
print("Monthly done")

Monthly done


In [102]:
country = df.groupby("Country").agg(
    total_revenue   =("Revenue",    "sum"),
    total_orders    =("InvoiceNo",  "nunique"),
    total_customers =("CustomerID", "nunique")
).reset_index().sort_values("total_revenue", ascending=False).reset_index(drop=True)
country["total_revenue"] = country["total_revenue"].round(2)
print("Country done")

Country done


In [103]:
pop = df.groupby(["StockCode", "Description"]).agg(
    total_qty      =("Quantity",   "sum"),
    total_revenue  =("Revenue",    "sum"),
    unique_buyers  =("CustomerID", "nunique"),
    avg_basket_qty =("Quantity",   "mean")
).reset_index()
pop["total_revenue"]  = pop["total_revenue"].round(2)
pop["avg_basket_qty"] = pop["avg_basket_qty"].round(2)
pop = pop.sort_values("total_revenue", ascending=False).reset_index(drop=True)
print("Products done")


Products done


In [104]:
snapshot = pd.Timestamp("2011-12-10")
rfm = df.groupby("CustomerID").agg(
    last_purchase =("InvoiceDate_ISO", "max"),
    frequency     =("InvoiceNo",       "nunique"),
    monetary      =("Revenue",         "sum")
).reset_index()
rfm["recency"]  = (snapshot - rfm["last_purchase"]).dt.days
rfm["monetary"] = rfm["monetary"].round(2)
rfm = rfm.drop(columns=["last_purchase"])

rfm["R"] = pd.qcut(rfm["recency"],                        q=5, labels=[5,4,3,2,1]).astype(int)
rfm["F"] = pd.qcut(rfm["frequency"].rank(method="first"), q=5, labels=[1,2,3,4,5]).astype(int)
rfm["M"] = pd.qcut(rfm["monetary"].rank(method="first"),  q=5, labels=[1,2,3,4,5]).astype(int)
rfm["rfm_score"] = rfm["R"] + rfm["F"] + rfm["M"]

def assign_segment(row):
    r, f, m = row["R"], row["F"], row["M"]
    if r >= 4 and f >= 4 and m >= 4: return "Champions"
    if r >= 3 and f >= 3:            return "Loyal"
    if r <= 2 and f >= 3 and m >= 3: return "At Risk"
    return "Others"

rfm["segment"] = rfm.apply(assign_segment, axis=1)
print("RFM done:", rfm["segment"].value_counts().to_dict())

RFM done: {'Others': 1930, 'Loyal': 994, 'Champions': 953, 'At Risk': 461}


In [105]:
rfm["ab_group"] = rfm["segment"].apply(
    lambda s: "Treatment" if s in ["Champions", "Loyal"] else "Control"
)
ab = rfm.groupby("ab_group").agg(
    sample_size          =("CustomerID", "count"),
    avg_revenue_per_user =("monetary",   "mean"),
    total_revenue        =("monetary",   "sum")
).reset_index().rename(columns={"ab_group": "test_group"})
ab["avg_revenue_per_user"] = ab["avg_revenue_per_user"].round(2)
ab["total_revenue"]        = ab["total_revenue"].round(2)
control_avg   = ab.loc[ab["test_group"] == "Control",   "avg_revenue_per_user"].values[0]
treatment_avg = ab.loc[ab["test_group"] == "Treatment", "avg_revenue_per_user"].values[0]
uplift        = round((treatment_avg - control_avg) / control_avg * 100, 2)
ab["uplift_pct"] = ab["test_group"].apply(lambda x: uplift if x == "Treatment" else 0.0)
print("AB test done")

AB test done


In [106]:
rfm[["CustomerID","recency","frequency","monetary",
     "R","F","M","rfm_score","segment"]].to_csv("rfm_segments.csv",     index=False)
pop.to_csv("product_popularity.csv",  index=False)
monthly.to_csv("monthly_revenue.csv", index=False)
country.to_csv("country_revenue.csv", index=False)
ab.to_csv("ab_test_summary.csv",      index=False)
kpi.to_csv("kpi_summary.csv",         index=False)
print("\nAll 6 CSVs saved")



All 6 CSVs saved


In [ ]:
for fname in ["rfm_segments.csv", "product_popularity.csv", "monthly_revenue.csv",
              "country_revenue.csv", "ab_test_summary.csv", "kpi_summary.csv"]:
    files.download(fname)

print("Downloads triggered")

In [107]:
customer_products = df.groupby("CustomerID")["Description"].apply(set)

In [108]:
print("Building co-purchase matrix...")
co_purchase = defaultdict(int)
for products in customer_products:
    for a, b in combinations(sorted(products), 2):
        co_purchase[(a, b)] += 1

Building co-purchase matrix...


In [109]:
print("Total pairs:", len(co_purchase))

Total pairs: 4389649


In [110]:
print("Building product lookup...")
product_recs = defaultdict(list)
for (a, b), count in co_purchase.items():
    product_recs[a].append((b, count))
    product_recs[b].append((a, count))


Building product lookup...


In [111]:
for p in product_recs:
    product_recs[p] = sorted(product_recs[p], key=lambda x: x[1], reverse=True)

In [112]:
print("Lookup built for", len(product_recs), "products")

Lookup built for 3877 products


In [113]:
print("Building customer recommendations...")
rows = []
for cid in customer_products.index:
    bought = customer_products[cid]
    scores = defaultdict(int)
    for product in bought:
        for rec_product, score in product_recs.get(product, [])[:20]:
            if rec_product not in bought:
                scores[rec_product] += score
    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:5]
    for rank, (prod, score) in enumerate(ranked, 1):
        rows.append({
            "CustomerID":          cid,
            "rank":                rank,
            "recommended_product": prod,
            "score":               score,
            "method":              "rule_based"
        })

rule_recs = pd.DataFrame(rows)
print("Done. Shape:", rule_recs.shape)
print(rule_recs.head(10))

Building customer recommendations...
Done. Shape: (21690, 5)
   CustomerID  rank                 recommended_product  score      method
0       12346     1       LARGE CERAMIC TOP STORAGE JAR    102  rule_based
1       12346     2      SMALL CERAMIC TOP STORAGE JAR      88  rule_based
2       12346     3   SET OF 3 CAKE TINS PANTRY DESIGN      74  rule_based
3       12346     4        SET OF 4 PANTRY JELLY MOULDS     67  rule_based
4       12346     5              JAM MAKING SET PRINTED     66  rule_based
5       12347     1     PACK OF 72 RETROSPOT CAKE CASES   4491  rule_based
6       12347     2  WHITE HANGING HEART T-LIGHT HOLDER   3732  rule_based
7       12347     3                       PARTY BUNTING   3523  rule_based
8       12347     4   SET OF 3 CAKE TINS PANTRY DESIGN    3372  rule_based
9       12347     5             JUMBO BAG RED RETROSPOT   3027  rule_based


In [ ]:
rule_recs.to_csv("rule_based_recommendations.csv", index=False)
files.download("rule_based_recommendations.csv")
print("Downloaded.")

In [62]:
print("Building user-item matrix...")
user_item = df.groupby(["CustomerID", "Description"])["Quantity"].sum().unstack(fill_value=0)
print("Matrix shape:", user_item.shape)

Building user-item matrix...
Matrix shape: (4338, 3877)


In [63]:
matrix = csr_matrix(user_item.values)

In [64]:
print("Fitting KNN model...")
model = NearestNeighbors(metric="cosine", algorithm="brute", n_neighbors=11)
model.fit(matrix)
print("Model fitted.")

Fitting KNN model...
Model fitted.


In [65]:
customer_ids = list(user_item.index)
product_ids  = list(user_item.columns)
cust_to_idx  = {cid: i for i, cid in enumerate(customer_ids)}

In [66]:
def recommend_collab(customer_id, top_n=5):
    if customer_id not in cust_to_idx:
        return []

    idx = cust_to_idx[customer_id]
    distances, indices = model.kneighbors(matrix[idx], n_neighbors=11)

    # similar customers (skip first — that's the customer themselves)
    similar_idxs = indices.flatten()[1:]
    similar_dists = distances.flatten()[1:]

    bought = set(user_item.columns[(user_item.iloc[idx] > 0)])

    scores = defaultdict(float)
    for sim_idx, dist in zip(similar_idxs, similar_dists):
        similarity = 1 - dist
        sim_customer_row = user_item.iloc[sim_idx]
        sim_bought = sim_customer_row[sim_customer_row > 0].index
        for product in sim_bought:
            if product not in bought:
                scores[product] += similarity * sim_customer_row[product]

    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return ranked[:top_n]

In [67]:
for cid in [17850, 12347, 15311]:
    print(f"\nCustomer {cid} — collab filtering recommendations:")
    recs = recommend_collab(cid)
    for i, (prod, score) in enumerate(recs, 1):
        print(f"  {i}. {prod} (score: {score:.2f})")


Customer 17850 — collab filtering recommendations:
  1. RED HANGING HEART T-LIGHT HOLDER (score: 52.43)
  2. T-LIGHT HOLDER SWEETHEART HANGING (score: 17.74)
  3. WORLD WAR 2 GLIDERS ASSTD DESIGNS (score: 17.07)
  4. HOME BUILDING BLOCK WORD (score: 15.40)
  5. DECORATIVE WICKER HEART SMALL (score: 15.16)

Customer 12347 — collab filtering recommendations:
  1. DOUGHNUT LIP GLOSS  (score: 77.46)
  2. 10 COLOUR SPACEBOY PEN (score: 38.08)
  3. GROW A FLYTRAP OR SUNFLOWER IN TIN (score: 37.54)
  4. PACK OF 12 SPACEBOY TISSUES (score: 33.20)
  5. LUNCH BAG RED RETROSPOT (score: 28.88)

Customer 15311 — collab filtering recommendations:
  1. PARTY BUNTING (score: 361.62)
  2. JUMBO BAG PEARS (score: 297.05)
  3. JUMBO BAG VINTAGE LEAF (score: 270.18)
  4. JUMBO BAG VINTAGE DOILY  (score: 211.52)
  5. 60 TEATIME FAIRY CAKE CASES (score: 165.01)


In [68]:
print("\nBuilding recommendations for all customers...")
rows = []
for cid in customer_ids:
    recs = recommend_collab(cid, top_n=5)
    for rank, (prod, score) in enumerate(recs, 1):
        rows.append({
            "CustomerID":          cid,
            "rank":                rank,
            "recommended_product": prod,
            "score":               round(score, 4),
            "method":              "collaborative"
        })

collab_recs = pd.DataFrame(rows)
print("Done. Shape:", collab_recs.shape)
print(collab_recs.head(10))



Building recommendations for all customers...
Done. Shape: (21690, 5)
   CustomerID  rank                 recommended_product    score  \
0       12346     1      SMALL CERAMIC TOP STORAGE JAR   88.7471   
1       12346     2                  BLUE PUDDING SPOON  38.7391   
2       12346     3                   RED PUDDING SPOON  38.7391   
3       12346     4               JAM JAR WITH PINK LID  31.1834   
4       12346     5       LARGE CERAMIC TOP STORAGE JAR  28.5406   
5       12347     1                 DOUGHNUT LIP GLOSS   77.4629   
6       12347     2              10 COLOUR SPACEBOY PEN  38.0808   
7       12347     3  GROW A FLYTRAP OR SUNFLOWER IN TIN  37.5372   
8       12347     4         PACK OF 12 SPACEBOY TISSUES  33.1988   
9       12347     5             LUNCH BAG RED RETROSPOT  28.8846   

          method  
0  collaborative  
1  collaborative  
2  collaborative  
3  collaborative  
4  collaborative  
5  collaborative  
6  collaborative  
7  collaborative  
8  collab

In [ ]:
collab_recs.to_csv("collab_recommendations.csv", index=False)
files.download("collab_recommendations.csv")
print("Downloaded.")

In [69]:
rule_df  = rule_recs.rename(columns={"recommended_product": "rule_product",  "score": "rule_score"})
collab_df = collab_recs.rename(columns={"recommended_product": "collab_product", "score": "collab_score"})

rule_df   = rule_df[["CustomerID", "rank", "rule_product",   "rule_score"]]
collab_df = collab_df[["CustomerID", "rank", "collab_product", "collab_score"]]

In [70]:
comparison = rule_df.merge(collab_df, on=["CustomerID", "rank"])

In [71]:
comparison["same_recommendation"] = comparison["rule_product"] == comparison["collab_product"]

In [72]:
print("=== Overall Agreement ===")
agree_pct = comparison["same_recommendation"].mean() * 100
print(f"Same recommendation: {agree_pct:.1f}% of the time")

=== Overall Agreement ===
Same recommendation: 1.8% of the time


In [73]:
print("\n=== Agreement by Rank ===")
print(comparison.groupby("rank")["same_recommendation"].mean().apply(lambda x: f"{x*100:.1f}%"))



=== Agreement by Rank ===
rank
1    4.5%
2    1.6%
3    1.2%
4    0.7%
5    0.7%
Name: same_recommendation, dtype: object


In [74]:
rule_coverage   = rule_recs.groupby("CustomerID")["rank"].count()
collab_coverage = collab_recs.groupby("CustomerID")["rank"].count()


In [75]:
print("\n=== Coverage (customers with full 5 recs) ===")
print(f"Rule-based:    {(rule_coverage == 5).sum()} / {len(rule_coverage)} customers")
print(f"Collaborative: {(collab_coverage == 5).sum()} / {len(collab_coverage)} customers")


=== Coverage (customers with full 5 recs) ===
Rule-based:    4338 / 4338 customers
Collaborative: 4338 / 4338 customers


In [76]:
print("\n=== Diversity (unique products recommended) ===")
print(f"Rule-based:    {rule_recs['recommended_product'].nunique()} unique products")
print(f"Collaborative: {collab_recs['recommended_product'].nunique()} unique products")



=== Diversity (unique products recommended) ===
Rule-based:    250 unique products
Collaborative: 1269 unique products


In [77]:
pop = pd.read_csv("product_popularity.csv")

In [78]:
rule_value = rule_recs.merge(
    pop[["Description", "total_revenue", "unique_buyers"]],
    left_on="recommended_product", right_on="Description", how="left"
)
collab_value = collab_recs.merge(
    pop[["Description", "total_revenue", "unique_buyers"]],
    left_on="recommended_product", right_on="Description", how="left"
)


In [79]:
print("\n=== Avg Revenue Potential of Recommended Products ===")
print(f"Rule-based:    £{rule_value['total_revenue'].mean():,.2f} avg product revenue")
print(f"Collaborative: £{collab_value['total_revenue'].mean():,.2f} avg product revenue")


=== Avg Revenue Potential of Recommended Products ===
Rule-based:    £62,890.04 avg product revenue
Collaborative: £12,967.88 avg product revenue


In [80]:
print("\n=== Avg Unique Buyers of Recommended Products (popularity) ===")
print(f"Rule-based:    {rule_value['unique_buyers'].mean():.1f} avg buyers")
print(f"Collaborative: {collab_value['unique_buyers'].mean():.1f} avg buyers")



=== Avg Unique Buyers of Recommended Products (popularity) ===
Rule-based:    670.6 avg buyers
Collaborative: 248.5 avg buyers


In [ ]:
comparison["agreement"] = comparison["same_recommendation"].map({True: "Same", False: "Different"})
comparison.to_csv("recommender_comparison.csv", index=False)
files.download("recommender_comparison.csv")
print("\nSaved and downloaded: recommender_comparison.csv")


Saved and downloaded: recommender_comparison.csv


In [82]:
summary = pd.DataFrame([
    {
        "method":                    "Rule-Based",
        "unique_products_recommended": rule_recs["recommended_product"].nunique(),
        "customers_covered":          rule_recs["CustomerID"].nunique(),
        "avg_product_revenue":        round(rule_value["total_revenue"].mean(), 2),
        "avg_unique_buyers":          round(rule_value["unique_buyers"].mean(), 1),
        "agreement_pct":              1.8,
        "approach":                   "Co-purchase frequency"
    },
    {
        "method":                    "Collaborative Filtering",
        "unique_products_recommended": collab_recs["recommended_product"].nunique(),
        "customers_covered":          collab_recs["CustomerID"].nunique(),
        "avg_product_revenue":        round(collab_value["total_revenue"].mean(), 2),
        "avg_unique_buyers":          round(collab_value["unique_buyers"].mean(), 1),
        "agreement_pct":              1.8,
        "approach":                   "Customer similarity (KNN cosine)"
    }
])

In [83]:
print(summary.T)


                                                 0  \
method                                  Rule-Based   
unique_products_recommended                    250   
customers_covered                             4338   
avg_product_revenue                       62890.04   
avg_unique_buyers                            670.6   
agreement_pct                                  1.8   
approach                     Co-purchase frequency   

                                                            1  
method                                Collaborative Filtering  
unique_products_recommended                              1269  
customers_covered                                        4338  
avg_product_revenue                                  12967.88  
avg_unique_buyers                                       248.5  
agreement_pct                                             1.8  
approach                     Customer similarity (KNN cosine)  


In [ ]:
summary.to_csv("method_comparison_summary.csv", index=False)
files.download("method_comparison_summary.csv")
print("\nDownloaded.")

In [85]:
products = pop[["Description", "total_revenue", "unique_buyers"]].drop_duplicates(subset="Description").reset_index(drop=True)

print("Total products:", len(products))


Total products: 3877


In [86]:
tfidf = TfidfVectorizer(stop_words="english")
tfidf_matrix = tfidf.fit_transform(products["Description"])
print("TF-IDF matrix shape:", tfidf_matrix.shape)

TF-IDF matrix shape: (3877, 1971)


In [87]:
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)
print("Similarity matrix shape:", cosine_sim.shape)

Similarity matrix shape: (3877, 3877)


In [88]:
prod_to_idx = {desc: i for i, desc in enumerate(products["Description"])}

In [89]:
def recommend_content_based_product(product, top_n=5):
    if product not in prod_to_idx:
        return []
    idx = prod_to_idx[product]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = [s for s in sim_scores if s[0] != idx][:top_n]
    return [(products["Description"].iloc[i], round(score, 4)) for i, score in sim_scores]


In [90]:
for p in ["WHITE HANGING HEART T-LIGHT HOLDER", "JUMBO BAG RED RETROSPOT", "REGENCY CAKESTAND 3 TIER"]:
    print(f"\nContent-based recs for: '{p}'")
    for i, (prod, score) in enumerate(recommend_content_based_product(p), 1):
        print(f"  {i}. {prod} (similarity: {score})")


Content-based recs for: 'WHITE HANGING HEART T-LIGHT HOLDER'
  1. PINK HANGING HEART T-LIGHT HOLDER (similarity: 0.8334)
  2. RED HANGING HEART T-LIGHT HOLDER (similarity: 0.8186)
  3. HANGING HEART ZINC T-LIGHT HOLDER (similarity: 0.768)
  4. CREAM HANGING HEART T-LIGHT HOLDER (similarity: 0.768)
  5. HEART T-LIGHT HOLDER (similarity: 0.7619)

Content-based recs for: 'JUMBO BAG RED RETROSPOT'
  1. VINTAGE DOILY JUMBO BAG RED  (similarity: 0.6469)
  2. LUNCH BAG RED RETROSPOT (similarity: 0.6109)
  3. JUMBO BAG VINTAGE CHRISTMAS  (similarity: 0.5951)
  4. RED RETROSPOT PICNIC BAG (similarity: 0.5861)
  5. JUMBO BAG STRAWBERRY (similarity: 0.5855)

Content-based recs for: 'REGENCY CAKESTAND 3 TIER'
  1. SWEETHEART CAKESTAND 3 TIER (similarity: 0.7416)
  2. CAKESTAND, 3 TIER, LOVEHEART (similarity: 0.6554)
  3. SWEETHEART 3 TIER CAKE STAND  (similarity: 0.3183)
  4. SET OF 3 REGENCY CAKE TINS (similarity: 0.3156)
  5. REGENCY TEA PLATE PINK (similarity: 0.3134)


In [91]:
print("\nBuilding content-based recs for all customers...")
rows = []
for cid in customer_products.index:
    bought = customer_products[cid]
    scores = defaultdict(float)
    for product in bought:
        recs = recommend_content_based_product(product, top_n=10)
        for rec_product, score in recs:
            if rec_product not in bought:
                scores[rec_product] += score
    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:5]
    for rank, (prod, score) in enumerate(ranked, 1):
        rows.append({
            "CustomerID":          cid,
            "rank":                rank,
            "recommended_product": prod,
            "score":               round(score, 4),
            "method":              "content_based"
        })


Building content-based recs for all customers...


In [92]:
content_recs = pd.DataFrame(rows)
print("Done. Shape:", content_recs.shape)
print(content_recs.head(10))

Done. Shape: (21690, 5)
   CustomerID  rank                  recommended_product   score  \
0       12346     1       SMALL CERAMIC TOP STORAGE JAR   0.7582   
1       12346     2        LARGE CERAMIC TOP STORAGE JAR  0.7552   
2       12346     3            RED RETROSPOT STORAGE JAR  0.5557   
3       12346     4          GLASS  SONGBIRD STORAGE JAR  0.4625   
4       12346     5           GLASS SONGBIRD STORAGE JAR  0.4625   
5       12347     1           ALARM CLOCK BAKELIKE IVORY  4.2093   
6       12347     2  60 GOLD AND SILVER FAIRY CAKE CASES  3.5226   
7       12347     3      60 CAKE CASES VINTAGE CHRISTMAS  3.4464   
8       12347     4   PACK OF 60 PINK PAISLEY CAKE CASES  3.4258   
9       12347     5  SET OF 60 PANTRY DESIGN CAKE CASES   3.0858   

          method  
0  content_based  
1  content_based  
2  content_based  
3  content_based  
4  content_based  
5  content_based  
6  content_based  
7  content_based  
8  content_based  
9  content_based  


In [93]:
content_recs.to_csv("content_based_recommendations.csv", index=False)
files.download("content_based_recommendations.csv")
print("Downloaded.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloaded.
